In [2]:
import findspark
import pandas
import time
import re

In [3]:

findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("LocalNotebookApp1") \
    .getOrCreate()

25/08/03 16:37:12 WARN Utils: Your hostname, testnode1 resolves to a loopback address: 127.0.1.1; using 192.168.1.144 instead (on interface eth0)
25/08/03 16:37:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/03 16:37:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/03 16:37:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/08/03 16:37:12 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [4]:

# === 1. Leer archivo normalmente con open() ===
file_path = "/var/snp-dwh/sftp/sftp-data/rvg_2004.txt"

start_total = time.time()  # ⏳ tiempo total desde el inicio
with open(file_path, "r", encoding="utf-8") as f:
        lineas = f.readlines()


In [5]:
# === 3. Crear RDD a partir de la lista de líneas ===
start_rdd = time.time()
rdd = spark.sparkContext.parallelize(lineas)

print(f"⏳ Tiempo paralelización: {time.time() - start_rdd:.2f} s")

# === 4. Procesamiento: limpiar palabras ===
def limpiar_palabra(p):
    return re.sub(r"[^a-zA-ZáéíóúÁÉÍÓÚñÑüÜ]", "", p).lower()

# Lista de palabras vacías comunes en español (opcional)
stopwords = {"y","de","que","el","la","en","a","los","se","del","un","por","con","no","una","su","al","lo","como","más","pero","sus","le","ya","o","este","sí","porque","esta","entre","cuando","muy","sin","sobre","también","me","hasta","hay","donde","quien","desde","todo","nos","durante","todos","uno","les","ni","contra","otros","ese","eso","ante","ellos","e","esto","mí","antes","algunos","qué","unos","yo","otro","otras","otra"}

# === 5. Conteo de palabras con Spark ===
start_job = time.time()
word_counts = (
    rdd.flatMap(lambda line: line.strip().split(" "))            
       .map(lambda word: limpiar_palabra(word))                  
       .filter(lambda w: w != "" and w not in stopwords)         
       .map(lambda word: (word, 1))
       .reduceByKey(lambda a, b: a + b)
)
print(f"⏳ Tiempo transformaciones Spark: {time.time() - start_job:.2f} s")

# === 6. Mostrar las 20 palabras más frecuentes ===
start_action = time.time()
top20 = word_counts.takeOrdered(20, key=lambda x: -x[1])
print(f"⏳ Tiempo acción (takeOrdered): {time.time() - start_action:.2f} s")

print("\n🔝 Top 20 palabras más frecuentes:")
for palabra, cantidad in top20:
    print(f"{palabra}: {cantidad}")

# === 7. Finalizar Spark ===
spark.stop()

# === 8. Tiempo total ===
print(f"\n✅ Tiempo total ejecución: {time.time() - start_total:.2f} s")


⏳ Tiempo paralelización: 0.14 s
⏳ Tiempo transformaciones Spark: 0.04 s


⏳ Tiempo acción (takeOrdered): 0.92 s

🔝 Top 20 palabras más frecuentes:
jehová: 6853
las: 5805
para: 5692
él: 4294
dios: 4131
tu: 3681
es: 3514
mi: 3404
tierra: 2929
dijo: 2871
hijos: 2842
israel: 2561
rey: 2504
te: 2421
hijo: 2405
he: 2345
mas: 2102
casa: 1966
entonces: 1827
pueblo: 1797

✅ Tiempo total ejecución: 2.11 s
